# Lesson 02 - Exploring Microsoft Agent Framework

The **Microsoft Agent Framework (MAF)** is a unified framework for building AI agents. It provides a clean, composable architecture with four core building blocks:

- **Client** – connects to an AI model endpoint and handles communication
- **Agent** – wraps a client with instructions and tool definitions
- **Tools** – extend agent capabilities with custom functions the model can call
- **Session** – maintains conversation history for multi-turn interactions

In this lesson, we'll build a **travel booking agent** that checks destination availability using these concepts.


## Configurazione


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Comprendere l'Architettura del Framework Agent

Il Microsoft Agent Framework segue un'architettura a strati:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – Un `FoundryChatClient` si collega a un deployment Azure OpenAI. Gestisce l'autenticazione, la formattazione delle richieste e l'analisi delle risposte.
2. **Agente** – Creato dal client tramite `provider.create_agent()`, l'agente combina l'accesso al modello con le istruzioni (prompt di sistema) e gli strumenti.
3. **Strumenti** – Funzioni Python decorate con `@tool` che l'agente può invocare per eseguire azioni o recuperare dati.
4. **Sessione** – Un oggetto `AgentSession` (creato tramite `agent.create_session()`) che memorizza la cronologia della conversazione, permettendo un dialogo multi-turno dove l'agente ricorda il contesto precedente.

Costruiamo ogni strato passo dopo passo.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Aggiungere Strumenti con il Decoratore @tool

Gli strumenti permettono agli agenti di effettuare azioni oltre alla generazione di testo. Il decoratore `@tool` trasforma una funzione Python normale in qualcosa che l'agente può chiamare.

Punti chiave:
- Usa `Annotated[type, "description"]` affinché il modello comprenda ogni parametro.
- La docstring diventa la descrizione dello strumento che il modello vede.
- `approval_mode="never_require"` significa che lo strumento si esegue automaticamente senza conferma dell'utente.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Creare un Agente con Strumenti

Ora combiniamo il client, le istruzioni e gli strumenti in un agente. Le `instructions` fungono da prompt di sistema — definiscono la personalità e il comportamento dell'agente.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Conversazioni Multi-Turn con le Sessioni

Una `AgentSession` (creata tramite `agent.create_session()`) tiene traccia di tutti i messaggi in una conversazione. Passando la stessa sessione a ogni chiamata `agent.run()`, l'agente ha accesso a tutta la cronologia della conversazione e può fare riferimento ai messaggi precedenti.

Passiamo `tools=[check_destination_availability]` così l'agente può chiamare il nostro verificatore di disponibilità durante ogni turno.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Sommario

In questa lezione hai esplorato i quattro pilastri del Microsoft Agent Framework:

| Concept | Cosa Hai Imparato |
|---------|------------------|
| **Client** | `FoundryChatClient` si connette a Azure OpenAI con autenticazione basata su credenziali |
| **Agent** | `provider.create_agent()` combina una connessione modello con istruzioni e un nome |
| **Tools** | Il decoratore `@tool` rende accessibili funzioni Python per l'agente da chiamare |
| **Session** | `agent.create_session()` mantiene la cronologia della conversazione attraverso molteplici turni |

Questi elementi costitutivi si combinano per creare agenti che possono sostenere conversazioni naturali, chiamare funzioni esterne e mantenere il contesto — la base per schemi agentici più avanzati nelle lezioni successive.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
